- Import required libraries

In [3]:
# Importing the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


- Import dataset

In [4]:
df = pd.read_csv('customer_behavior_history.csv', parse_dates=['signup_date', 'event_timestamp'])
df = df.drop(columns=['key','device_id', 'device_category', 'signup_date', 'watch_time', 'amount'])
df.tail()


,member_id,device_name,partner_name,country,region,event_type,event_status,event_timestamp
119995,1552,Amazon Fire TV,Samsung,US,EMEA,billing,Success,2024-07-04 19:38:26.819164
119996,4394,PS5,Google,VN,Americas,billing,Failed,2025-07-31 08:02:36.006365
119997,4410,iPad,Apple,CA,Americas,signup,Failed,2025-02-28 21:36:52.395120
119998,1082,iPad,Amazon,CA,APAC,signup,Failed,2025-02-11 21:39:01.224203
119999,1832,PS5,Apple,FR,Americas,playback,Failed,2026-03-18 23:53:18.511336


- Keep only retention event

In [5]:
df = df[(df['event_type'] == 'billing') & (df['event_status']== 'Success')].copy()
df.tail()


,member_id,device_name,partner_name,country,region,event_type,event_status,event_timestamp
119989,2722,Amazon Fire TV,Apple,FR,APAC,billing,Success,2024-09-06 20:16:20.062171
119991,3927,PS5,Google,CA,EMEA,billing,Success,2024-06-23 19:49:18.419641
119993,2210,iPad,Google,ES,EMEA,billing,Success,2026-01-13 19:25:43.430291
119994,2954,Chromecast,Sony,FR,EMEA,billing,Success,2025-12-27 09:16:50.361062
119995,1552,Amazon Fire TV,Samsung,US,EMEA,billing,Success,2024-07-04 19:38:26.819164


- Create a cohort group for each customer based on their first purchase date
- Transform the event_timestamp to extract the month period

In [6]:
df['first_event'] = (
    df.groupby('member_id')['event_timestamp']
    .transform('min')
)

df['cohort_first_month'] = (
    df['first_event']
    .dt.to_period('M')
)

df.tail()

,member_id,device_name,partner_name,country,region,event_type,event_status,event_timestamp,first_event,cohort_first_month
119989,2722,Amazon Fire TV,Apple,FR,APAC,billing,Success,2024-09-06 20:16:20.062171,2024-09-06 20:16:20.062171,2024-09
119991,3927,PS5,Google,CA,EMEA,billing,Success,2024-06-23 19:49:18.419641,2024-05-20 06:22:54.237298,2024-05
119993,2210,iPad,Google,ES,EMEA,billing,Success,2026-01-13 19:25:43.430291,2024-12-12 12:07:27.155941,2024-12
119994,2954,Chromecast,Sony,FR,EMEA,billing,Success,2025-12-27 09:16:50.361062,2025-12-27 09:16:50.361062,2025-12
119995,1552,Amazon Fire TV,Samsung,US,EMEA,billing,Success,2024-07-04 19:38:26.819164,2024-04-14 15:20:24.151354,2024-04


- Transform the event_timestamp to extract the month period

In [7]:
df['first_event'] = (
    df.groupby('member_id')['event_timestamp']
    .transform('min')
)

df['cohort_first_month'] = (
    df['first_event']
    .dt.to_period('M')
)
df['event_month'] = df['event_timestamp'].dt.to_period('M')
df = df[(df['event_month'] <= pd.Period('2025-12', freq='M')) 
            & (df['event_month'] >= pd.Period('2025-01', freq='M'))
            & (df['cohort_first_month'] <= pd.Period('2025-12', freq='M')) 
            & (df['cohort_first_month'] >= pd.Period('2025-01', freq='M'))]
df.tail()

,member_id,device_name,partner_name,country,region,event_type,event_status,event_timestamp,first_event,cohort_first_month,event_month
119798,3352,Amazon Fire TV,Samsung,ES,Americas,billing,Success,2025-02-12 05:12:08.664034,2025-02-12 05:12:08.664034,2025-02,2025-02
119827,3338,Chromecast,Amazon,CA,Americas,billing,Success,2025-03-18 01:28:50.435612,2025-03-18 01:28:50.435612,2025-03,2025-03
119875,2329,PS5,Google,ES,APAC,billing,Success,2025-12-15 14:04:45.391062,2025-02-27 11:40:13.990470,2025-02,2025-12
119901,1247,Chromecast,Google,DE,EMEA,billing,Success,2025-10-13 18:46:03.212665,2025-10-01 23:17:06.674462,2025-10,2025-10
119994,2954,Chromecast,Sony,FR,EMEA,billing,Success,2025-12-27 09:16:50.361062,2025-12-27 09:16:50.361062,2025-12,2025-12


In [8]:
df.dtypes

member_id                      int64
device_name                      str
partner_name                     str
country                          str
region                           str
event_type                       str
event_status                     str
event_timestamp       datetime64[us]
first_event           datetime64[us]
cohort_first_month         period[M]
event_month                period[M]
dtype: object

- Create the number of retention months

In [9]:

df['month_diff'] = (
    df['event_month'] - df['cohort_first_month']
).apply(lambda x: x.n)


df.tail()


,member_id,device_name,partner_name,country,region,event_type,event_status,event_timestamp,first_event,cohort_first_month,event_month,month_diff
119798,3352,Amazon Fire TV,Samsung,ES,Americas,billing,Success,2025-02-12 05:12:08.664034,2025-02-12 05:12:08.664034,2025-02,2025-02,0
119827,3338,Chromecast,Amazon,CA,Americas,billing,Success,2025-03-18 01:28:50.435612,2025-03-18 01:28:50.435612,2025-03,2025-03,0
119875,2329,PS5,Google,ES,APAC,billing,Success,2025-12-15 14:04:45.391062,2025-02-27 11:40:13.990470,2025-02,2025-12,10
119901,1247,Chromecast,Google,DE,EMEA,billing,Success,2025-10-13 18:46:03.212665,2025-10-01 23:17:06.674462,2025-10,2025-10,0
119994,2954,Chromecast,Sony,FR,EMEA,billing,Success,2025-12-27 09:16:50.361062,2025-12-27 09:16:50.361062,2025-12,2025-12,0


In [10]:
cohort_size = df[df['month_diff'] == 0].groupby('cohort_first_month')['member_id'].nunique().rename('cohort_size')

- Similar pratice in Looker Studio (Data Studio)

In [11]:
cohort_data = (
    df.groupby(['cohort_first_month', 'month_diff'])['member_id']
    .nunique()
    .reset_index()
    .rename(columns={'member_id': 'retained_users'})
)

cohort_data = cohort_data.join(cohort_size, on='cohort_first_month')
cohort_data['retention_rate'] = (
    cohort_data['retained_users'] / cohort_data['cohort_size'] * 100
).round(1)

In [12]:
cohort_data

,cohort_first_month,month_diff,retained_users,cohort_size,retention_rate
0,2025-01,0,110,110,100.0
1,2025-01,1,15,110,13.6
2,2025-01,2,19,110,17.3
3,2025-01,3,15,110,13.6
4,2025-01,4,18,110,16.4
...,...,...,...,...,...
73,2025-10,1,1,13,7.7
74,2025-10,2,2,13,15.4
75,2025-11,0,12,12,100.0
76,2025-11,1,3,12,25.0


In [13]:
cohort_pivot_percent = cohort_data.pivot_table(
    index='cohort_first_month',
    columns='month_diff',
    values='retention_rate'
)
cohort_pivot_percent.index = cohort_pivot_percent.index.astype(str)

In [14]:
cohort_pivot_absolute = cohort_data.pivot_table(
    index='cohort_first_month',
    columns='month_diff',
    values='retained_users'
    )
cohort_pivot_absolute.index = cohort_pivot_absolute.index.astype(str)

In [15]:
cohort_pivot_percent

month_diff,0,1,2,3,4,5,6,7,8,9,10,11
cohort_first_month,,,,,,,,,,,,
2025-01,100.0,13.6,17.3,13.6,16.4,19.1,21.8,15.5,16.4,20.0,18.2,12.7
2025-02,100.0,19.8,9.9,22.0,15.4,17.6,20.9,22.0,14.3,18.7,20.9,NaN
2025-03,100.0,18.2,18.2,24.7,22.1,20.8,18.2,19.5,26.0,20.8,NaN,NaN
2025-04,100.0,23.5,14.7,16.2,23.5,23.5,11.8,14.7,17.6,NaN,NaN,NaN
2025-05,100.0,18.0,18.0,24.6,11.5,13.1,9.8,13.1,NaN,NaN,NaN,NaN
2025-06,100.0,17.9,10.3,20.5,25.6,25.6,10.3,NaN,NaN,NaN,NaN,NaN
2025-07,100.0,20.7,13.8,17.2,13.8,20.7,NaN,NaN,NaN,NaN,NaN,NaN
2025-08,100.0,18.4,13.2,15.8,23.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,100.0,14.3,17.9,14.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
cohort_pivot_absolute

month_diff,0,1,2,3,4,5,6,7,8,9,10,11
cohort_first_month,,,,,,,,,,,,
2025-01,110.0,15.0,19.0,15.0,18.0,21.0,24.0,17.0,18.0,22.0,20.0,14.0
2025-02,91.0,18.0,9.0,20.0,14.0,16.0,19.0,20.0,13.0,17.0,19.0,NaN
2025-03,77.0,14.0,14.0,19.0,17.0,16.0,14.0,15.0,20.0,16.0,NaN,NaN
2025-04,68.0,16.0,10.0,11.0,16.0,16.0,8.0,10.0,12.0,NaN,NaN,NaN
2025-05,61.0,11.0,11.0,15.0,7.0,8.0,6.0,8.0,NaN,NaN,NaN,NaN
2025-06,39.0,7.0,4.0,8.0,10.0,10.0,4.0,NaN,NaN,NaN,NaN,NaN
2025-07,29.0,6.0,4.0,5.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
2025-08,38.0,7.0,5.0,6.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,28.0,4.0,5.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Visualization

In [17]:
# =========================================================
# Dark theme
# =========================================================

DARK_BG = "#1e1e1e"
GRID_COLOR = "#2a2a2a"
TEXT_COLOR = "#ffffff"


def apply_dark_theme(fig):

    fig.update_layout(
        paper_bgcolor=DARK_BG,
        plot_bgcolor=DARK_BG,

        font=dict(
            color=TEXT_COLOR,
            size=12
        ),

        margin=dict(
            l=70,
            r=40,
            t=80,
            b=60
        ),

        hoverlabel=dict(
            bgcolor="#2a2a2a",
            font=dict(
                color="white"
            ),
            bordercolor="#444444"
        ),

        autosize=True,

        dragmode="pan"
    )

    fig.update_xaxes(
        color=TEXT_COLOR,
        gridcolor=GRID_COLOR,
        linecolor=GRID_COLOR,
        zeroline=False
    )

    fig.update_yaxes(
        color=TEXT_COLOR,
        gridcolor=GRID_COLOR,
        linecolor=GRID_COLOR,
        zeroline=False
    )

    return fig


# =========================================================
# Dynamic Cohort Heatmap
# =========================================================

def plot_dynamic_heatmap(
    data,
    is_percent=False,
    title=None
):

    # Make a copy so the original dataframe is untouched
    data = data.copy()

    # -----------------------------------------------------
    # Format values shown inside cells
    # -----------------------------------------------------

    if is_percent:

        text_values = data.map(
            lambda x: f"{x:.1f}%"
            if pd.notna(x)
            else ""
        )

        hover_values = data.map(
            lambda x: f"{x:.1f}%"
            if pd.notna(x)
            else "No data"
        )

    else:

        text_values = data.map(
            lambda x: f"{x:,.0f}"
            if pd.notna(x)
            else ""
        )

        hover_values = data.map(
            lambda x: f"{x:,.0f}"
            if pd.notna(x)
            else "No data"
        )

    # -----------------------------------------------------
    # Create heatmap
    # -----------------------------------------------------

    fig = go.Figure(

        data=go.Heatmap(

            z=data.values,

            x=data.columns.astype(str),

            y=data.index.astype(str),

            colorscale="Blues",

            # Values displayed inside cells
            text=text_values.values,

            texttemplate="%{text}",

            textfont=dict(
                color="black",
                size=10
            ),

            # Information displayed on hover
            customdata=hover_values.values,

            hovertemplate=(
                "<b>Cohort:</b> %{y}<br>"
                "<b>Month:</b> %{x}<br>"
                "<b>Value:</b> %{customdata}"
                "<extra></extra>"
            ),

            # Cell spacing
            xgap=1,
            ygap=1,

            # Color scale
            colorbar=dict(

                title=dict(
                    text="Retention" if is_percent else "Users",

                    font=dict(
                        color=TEXT_COLOR
                    )
                ),

                tickfont=dict(
                    color=TEXT_COLOR
                )
            ),

            # Do not interpolate missing cohort periods
            connectgaps=False
        )
    )

    # -----------------------------------------------------
    # Axis configuration
    # -----------------------------------------------------

    fig.update_layout(

        title=dict(
            text=title if title else "",

            font=dict(
                color=TEXT_COLOR,
                size=18
            )
        ),

        xaxis=dict(

            title=dict(
                text="Period (Months Since First Purchase)",
                font=dict(color=TEXT_COLOR)
            ),

            side="top",

            fixedrange=False
        ),

        yaxis=dict(

            title=dict(
                text="Cohort Month",
                font=dict(color=TEXT_COLOR)
            ),

            # Oldest cohort at the top
            autorange="reversed",

            fixedrange=False
        )
    )

    # Apply dark theme
    apply_dark_theme(fig)

    return fig

In [18]:
plot_dynamic_heatmap(
    cohort_pivot_absolute,
    is_percent=False
)

In [23]:
plot_dynamic_heatmap(
    cohort_pivot_percent,
    is_percent=True,
    title="Cohort Retention Rate"
)

In [ ]:
# fig = plot_dynamic_heatmap(
#     cohort_pivot_percent,
#     is_percent=True
# )

# fig.write_html(
#     "cohort_pivot_percent.html",
#     include_plotlyjs="cdn",
#     full_html=True
# )

In [ ]:
# fig = plot_dynamic_heatmap(
#     cohort_pivot_absolute,
#     is_percent=False
# )

# fig.write_html(
#     "cohort_pivot_absolute_new.html",
#     include_plotlyjs="cdn",
#     full_html=True
# )

In [24]:
def plot_dynamic_cohort_curve(
    data,
    title=None
):

    data = data.copy()

    fig = go.Figure()

    # Generate colors
    colors = px.colors.sequential.Blues

    n_cohorts = len(data.index)

    for i, cohort in enumerate(data.index):

        # Spread cohorts across the color scale
        color_index = int(
            i / max(n_cohorts - 1, 1)
            * (len(colors) - 1)
        )

        color = colors[color_index]

        fig.add_trace(

            go.Scatter(

                x=data.columns,

                y=data.loc[cohort],

                mode="lines+markers",

                name=str(cohort),

                line=dict(
                    color=color,
                    width=2
                ),

                marker=dict(
                    size=6
                ),

                hovertemplate=(
                    "<b>Cohort:</b> %{fullData.name}<br>"
                    "<b>Month:</b> %{x}<br>"
                    "<b>Retention:</b> %{y:.1f}%"
                    "<extra></extra>"
                )
            )
        )

    fig.update_layout(

        title=dict(
            text=title if title else "",

            font=dict(
                color=TEXT_COLOR,
                size=18
            )
        ),

        xaxis=dict(

            title=dict(
                text="Months Since First Purchase",
                font=dict(color=TEXT_COLOR)
            ),

            dtick=1,

            fixedrange=False
        ),

        yaxis=dict(

            title=dict(
                text="Retention Rate",
                font=dict(color=TEXT_COLOR)
            ),

            ticksuffix="%",

            fixedrange=False
        ),

        hovermode="closest",

        dragmode="pan",

        legend=dict(

            title=dict(
                text="Cohort",
                font=dict(color=TEXT_COLOR)
            ),

            font=dict(
                color=TEXT_COLOR
            ),

            bgcolor="rgba(0,0,0,0)",

            orientation="h",

            yanchor="bottom",
            y=1.02,

            xanchor="left",
            x=0
        )
    )

    apply_dark_theme(fig)

    return fig

In [26]:
plot_dynamic_cohort_curve(
    cohort_pivot_percent,
)

In [ ]:
# fig = plot_dynamic_cohort_curve(
#     cohort_pivot_percent,
# )

# fig.write_html(
#     "cohort_curve_pivot_percent.html",
#     include_plotlyjs="cdn",
#     full_html=True
# )

In [19]:
# # Create dynamic interactive visualizations using Plotly

# # Dynamic heatmap for absolute user counts
# fig_absolute = plot_dynamic_heatmap(cohort_pivot_absolute, is_percent=False, title='User Count Heatmap')
# fig_absolute.show()

# # Dynamic heatmap for retention rates
# fig_percent = plot_dynamic_heatmap(cohort_pivot_percent, is_percent=True, title='Retention Rate Heatmap (%)')
# fig_percent.show()

# # Dynamic cohort retention curves
# fig_curves = plot_dynamic_cohort_curve(cohort_pivot_percent)
# fig_curves.show()

In [20]:
# # Dynamic retention rate heatmap using Plotly
# fig_percent = plot_dynamic_heatmap(cohort_pivot_percent, is_percent=True, title='Retention Rate Heatmap (%)')
# fig_percent.show()

In [21]:
# Dynamic user count heatmap using Plotly
fig_absolute = plot_dynamic_heatmap(cohort_pivot_absolute, is_percent=False, title='User Count Heatmap')
fig_absolute.show()

In [ ]:
# # Dynamic cohort retention curves using Plotly
# fig_curves = plot_dynamic_cohort_curve(cohort_pivot_percent)
# fig_curves.show()

NameError: name 'plot_dynamic_cohort_curve' is not defined